# 01 - Data exploration and baseline

**Goal:** look at the dataset, tag Hinglish rows, and get a simple TF-IDF + Logistic Regression baseline.
The baseline score is the number the transformers have to beat.

Labels: `0` = negative, `1` = neutral, `2` = positive.

In [2]:
%pip install -q ftfy tweet-preprocessor

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, sys, subprocess

REPO_URL = "https://github.com/23f2004336/hinglish-sentiment.git"

# Make `src/` importable: works locally (inside notebooks/ or the repo root) and on Kaggle.
if os.path.exists("../src"):
    sys.path.insert(0, "..")
    RESULTS_DIR = "../results"
elif os.path.exists("src"):
    sys.path.insert(0, ".")
    RESULTS_DIR = "results"
else:
    if not os.path.exists("hinglish-sentiment"):
        subprocess.run(["git", "clone", REPO_URL], check=True)   # needs Internet ON in Kaggle
    sys.path.insert(0, os.path.abspath("hinglish-sentiment"))
    RESULTS_DIR = "."

Cloning into 'hinglish-sentiment'...


## Load, clean and tag the data

In [4]:
import json
import pandas as pd
from src.data import load_data

dfs = load_data()
for split, df in dfs.items():
    print(f"{split:<11} {len(df):>6} rows | hinglish: {int(df.hinglish.sum())}")

train, val, test = dfs["train"], dfs["validation"], dfs["test"]

README.md:   0%|          | 0.00/533 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 22.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.92MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/96761 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8538 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8538 [00:00<?, ? examples/s]

train        96761 rows | hinglish: 5020
validation    8538 rows | hinglish: 476
test          8538 rows | hinglish: 485


## Label mix per slice

English is roughly balanced, but Hinglish is mostly neutral. That is why we report **Macro F1** and not accuracy
(always predicting "neutral" on Hinglish would already give about 58% accuracy).

In [5]:
print(train.groupby("hinglish").label.value_counts(normalize=True).round(2).unstack())

label        0     1     2
hinglish                  
False     0.36  0.28  0.36
True      0.27  0.58  0.15


## Spot-check the Hinglish tag
Read a few rows from each side. A few misses are fine.

In [6]:
print("TAGGED as Hinglish:")
for t in train[train.hinglish].text.sample(10, random_state=1): print("-", t[:120])
print("\nNOT tagged:")
for t in train[~train.hinglish].text.sample(10, random_state=1): print("-", t[:120])

TAGGED as Hinglish:
- chutiye fir logo ko chutiya bana raha hai 😂😂 fraud saale nakli evm ka dabba dikha kr logo k …
- kaun bola the modest banne .. marwa liye na wo bhi kamara jaise chutiye se ..
- sab ka vikas sab ka visvas congratulations hble pm sir
- congress + opposition ko jab news debate main jaake govt ko pelke hawa / nanga karna tha gdp + unemployment data pe ye …
- wo to hai hi ye sale atankwadiyon k yaar hafij masud jiosama ji wale party or khandan se .. sale rahul chutiye atank …
- jeet ki hardik badhai hume bharosa h aap par ki hamara desh duniya me no one par aapke netratav me aa …
- why women if all are equal than samanta ke adhikar ke tahat ye suvidha sabko milni chahiye nahi to kisi ki nahi .
- baat aise kartay hain pmln walay jaise super power chor kar gaye the pakistan ko
- jumma jumma aath din bhi nahi hue sarkar banke aur ye ek community ke neta phir hamesha ki tar …
- jo captain afg sy match nahi jeet sakta waho nz / aus / ind / eng etc ko to khwab me be nahi hara

## Baseline: TF-IDF + Logistic Regression

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000)
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(vec.fit_transform(train.text), train.label)

pred = clf.predict(vec.transform(test.text))
y, m = test.label.values, test.hinglish.values

res = {
    "model": "tfidf_logreg",
    "overall": f1_score(y, pred, average="macro"),
    "hinglish": f1_score(y[m], pred[m], average="macro"),
    "english": f1_score(y[~m], pred[~m], average="macro"),
}
print({k: (round(v, 3) if not isinstance(v, str) else v) for k, v in res.items()})
print("\nHinglish only:\n", classification_report(y[m], pred[m]))

with open(os.path.join(RESULTS_DIR, "baseline_results.json"), "w") as f:
    json.dump(res, f, indent=2)

{'model': 'tfidf_logreg', 'overall': 0.817, 'hinglish': 0.518, 'english': 0.826}

Hinglish only:
               precision    recall  f1-score   support

           0       0.78      0.32      0.46       134
           1       0.67      0.91      0.77       286
           2       0.44      0.26      0.33        65

    accuracy                           0.66       485
   macro avg       0.63      0.50      0.52       485
weighted avg       0.67      0.66      0.62       485



## Notes
- The Hinglish slice is small (about 485 test rows), so scores on it are noisy.
- Hinglish rows are almost all tweets; English rows are mostly reviews. So "Hinglish" here also means "tweets".